# VideoMAE

## Cell 1 — Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Same base folder as VideoSwin notebook.
# If your folder is somewhere else, change ONLY this line.
BASE = "/content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab"

DATA_DIR    = f"{BASE}/data"
VIDEO_DIR   = f"{BASE}/data/videos"
LABEL_DIR   = f"{BASE}/data/labels"
RESULTS_DIR = f"{BASE}/results"
LOG_DIR     = f"{BASE}/logs/videomae_aligned"

for d in [RESULTS_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('BASE:', BASE)
print('VIDEO_DIR:', VIDEO_DIR)
print('LABEL_DIR:', LABEL_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print("\nLabel files:")
!ls "$LABEL_DIR"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab
VIDEO_DIR: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/data/videos
LABEL_DIR: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/data/labels
RESULTS_DIR: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/results

Label files:
split_10class.json  split_174class.json  split_20class.json  split_50class.json


## Cell 2 — Install dependencies

In [ ]:
!pip install -q transformers accelerate decord pynvml scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 111.6 MB/s eta 0:00:00


## Cell 3 — Imports, GPU check, config

In [ ]:
import os, json, time, random, threading, inspect, math
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from PIL import Image
import cv2

from sklearn.metrics import accuracy_score
from transformers import (
    AutoImageProcessor,
    VideoMAEForVideoClassification,
    TrainingArguments,
    Trainer,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONFIG = {
    'splits': [10, 20, 50],
    'epochs': 5,
    'batch_size': 2,
    'grad_accum': 4,
    'num_frames': 16,
    'lr_full': 5e-5,
    'lr_lora': 1e-4,
    'weight_decay': 0.05,
    'lora_r': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.1,
    'seed': SEED,
}

MODEL_CKPT = 'MCG-NJU/videomae-base-finetuned-kinetics'
OUT_PATH = f'{RESULTS_DIR}/results_videomae_aligned.json'

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
else:
    print('WARNING: GPU is not available. Do not run training on CPU.')

print('CONFIG:', CONFIG)
print('OUT_PATH:', OUT_PATH)

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8
CONFIG: {'splits': [10, 20, 50], 'epochs': 5, 'batch_size': 2, 'grad_accum': 4, 'num_frames': 16, 'lr_full': 5e-05, 'lr_lora': 0.0001, 'weight_decay': 0.05, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1, 'seed': 42}
OUT_PATH: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/results/results_videomae_aligned.json


## Cell 4 — Load VideoMAE image processor

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(MODEL_CKPT)
print(image_processor)
print('Mean:', image_processor.image_mean)
print('Std:', image_processor.image_std)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

The image processor of type `VideoMAEImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


VideoMAEImageProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "VideoMAEImageProcessor",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

Mean: [0.485, 0.456, 0.406]
Std: [0.229, 0.224, 0.225]


## Cell 5 — Robust video reader

This is based on the previously working VideoMAE notebook. It returns a list of PIL images, which the HuggingFace image processor converts into the correct VideoMAE tensor format.

In [ ]:
def read_video_frames(video_path, num_frames=16, verbose=False):
    """Read evenly sampled frames as PIL images. Tries decord first, then OpenCV."""
    # Try decord first
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        if total_frames <= 0:
            raise ValueError('0 frames')
        indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
        frames = vr.get_batch(indices).asnumpy()
        return [Image.fromarray(frame) for frame in frames]
    except Exception as e:
        if verbose:
            print('Decord failed:', str(e)[:200])
            print('Trying OpenCV fallback...')

    # OpenCV fallback
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'OpenCV cannot open video: {video_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError(f'OpenCV found 0 frames: {video_path}')

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame))
    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames could be read from video: {video_path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return frames[:num_frames]

## Cell 6 — Load the exact same VideoSwin split JSONs

This is the most important fairness cell: VideoMAE reads the exact same split files that VideoSwin used.

In [ ]:
import os, json, random
from collections import Counter

def fix_video_path(path):
    """
    Convert old paths from split JSON:
    ./ssv2_test/Class/video.webm

    into current Drive paths:
    /content/drive/.../cs_lab/data/videos/Class/video.webm
    """

    # If original path already works, keep it
    if os.path.exists(path):
        return path

    # Main case: ./ssv2_test/Class/video.webm
    if "ssv2_test/" in path:
        rel = path.split("ssv2_test/", 1)[1]
        new_path = os.path.join(VIDEO_DIR, rel)
        return new_path

    # Backup case: some old full path containing /data/videos/
    if "/data/videos/" in path:
        rel = path.split("/data/videos/", 1)[1]
        new_path = os.path.join(VIDEO_DIR, rel)
        return new_path

    return path


def load_swin_split_for_videomae(split_n):
    split_path = f'{LABEL_DIR}/split_{split_n}class.json'

    if not os.path.exists(split_path):
        raise FileNotFoundError(f'Missing split file: {split_path}')

    with open(split_path, 'r') as f:
        data = json.load(f)

    classes = data['classes']
    label2id = {cls: i for i, cls in enumerate(classes)}
    id2label = {i: cls for cls, i in label2id.items()}

    train_samples = [(fix_video_path(s['path']), label2id[s['label']]) for s in data['train']]
    val_samples   = [(fix_video_path(s['path']), label2id[s['label']]) for s in data['val']]

    rng = random.Random(SEED)
    rng.shuffle(train_samples)

    print('=' * 60)
    print(f'Split: {split_n} classes')
    print('Classes:', len(classes))
    print('Train samples:', len(train_samples))
    print('Val samples:', len(val_samples))
    print('Train distribution:', Counter([y for _, y in train_samples]))
    print('Val distribution:', Counter([y for _, y in val_samples]))

    print('\nFirst train sample:')
    print(train_samples[0][0])
    print('Exists:', os.path.exists(train_samples[0][0]))

    missing = [p for p, _ in train_samples[:50] if not os.path.exists(p)]
    print('Missing among first 50 train samples:', len(missing))

    if missing:
        print('Example missing:', missing[0])

    return classes, label2id, id2label, train_samples, val_samples


classes, label2id, id2label, train_samples, val_samples = load_swin_split_for_videomae(10)

Split: 10 classes
Classes: 10
Train samples: 1148
Val samples: 287
Train distribution: Counter({2: 310, 8: 165, 9: 160, 1: 122, 4: 99, 5: 89, 6: 74, 3: 62, 7: 42, 0: 25})
Val distribution: Counter({2: 76, 9: 48, 8: 47, 1: 26, 4: 22, 5: 18, 0: 17, 6: 14, 3: 13, 7: 6})

First train sample:
/content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/data/videos/Spinning something so it continues spinning/40440.webm
Exists: True
Missing among first 50 train samples: 0


## Cell 7 — Dataset and collate function

In [ ]:
class VideoClassificationDataset(Dataset):
    def __init__(self, samples, image_processor, num_frames=16):
        self.samples = samples
        self.image_processor = image_processor
        self.num_frames = num_frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        try:
            frames = read_video_frames(video_path, num_frames=self.num_frames)
            inputs = self.image_processor(frames, return_tensors='pt')
            pixel_values = inputs['pixel_values'].squeeze(0)  # (T, C, H, W)
            return {
                'pixel_values': pixel_values,
                'labels': torch.tensor(label, dtype=torch.long),
            }
        except Exception as e:
            # Avoid crashing a long experiment due to one corrupted video.
            print(f'Error reading {video_path}: {e}')
            new_idx = random.randint(0, len(self.samples) - 1)
            return self.__getitem__(new_idx)


def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    return {'pixel_values': pixel_values, 'labels': labels}

# Quick tensor shape check
train_dataset = VideoClassificationDataset(train_samples[:8], image_processor, CONFIG['num_frames'])
item = train_dataset[0]
print('One item pixel_values:', item['pixel_values'].shape)  # expected (16, 3, 224, 224)
print('One item label:', item['labels'])

One item pixel_values: torch.Size([16, 3, 224, 224])
One item label: tensor(8)


## Cell 8 — Metrics: accuracy and top-5 accuracy

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)

    k = min(5, logits.shape[1])
    topk = np.argsort(logits, axis=1)[:, -k:]
    topk_acc = np.mean([labels[i] in topk[i] for i in range(len(labels))])

    return {
        'accuracy': acc,
        'top5_accuracy': topk_acc,
    }

## Cell 9 — GPU power, memory, latency logger

Same output fields as VideoSwin: latency, average power, peak memory, energy.

In [ ]:
import pynvml

class PowerMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.values = []
        self.running = False
        self.thread = None
        self.handle = None

    def start(self):
        self.values = []
        self.running = True
        if torch.cuda.is_available():
            try:
                pynvml.nvmlInit()
                self.handle = pynvml.nvmlDeviceGetHandleByIndex(0)
            except Exception as e:
                print('pynvml init failed:', e)
                self.handle = None
        self.t0 = time.perf_counter()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.thread = threading.Thread(target=self._loop)
        self.thread.daemon = True
        self.thread.start()

    def _loop(self):
        while self.running:
            if self.handle is not None:
                try:
                    self.values.append(pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0)
                except Exception:
                    pass
            time.sleep(self.interval)

    def stop(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        latency = time.perf_counter() - self.t0
        self.running = False
        if self.thread is not None:
            self.thread.join(timeout=2)
        avg_power = float(np.mean(self.values)) if len(self.values) else 0.0
        peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0
        return {
            'latency_s': round(latency, 2),
            'avg_power_w': round(avg_power, 2),
            'peak_memory_mb': round(peak_mem, 2),
            'energy_j': round(avg_power * latency, 2),
        }

print('PowerMonitor ready')

PowerMonitor ready


## Cell 10 — Lightweight custom LoRA for VideoMAE

For LoRA runs, the backbone is frozen, LoRA adapters are inserted into attention query/value linear layers, and the classifier is trainable.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r=8, alpha=16, dropout=0.1):
        super().__init__()
        self.base_layer = base_layer
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        self.lora_dropout = nn.Dropout(dropout)

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        self.lora_A = nn.Linear(in_features, r, bias=False)
        self.lora_B = nn.Linear(r, out_features, bias=False)

        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)

        # Freeze original layer
        for p in self.base_layer.parameters():
            p.requires_grad = False

    @property
    def weight(self):
        return self.base_layer.weight

    @property
    def bias(self):
        return self.base_layer.bias

    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = self.lora_B(self.lora_A(self.lora_dropout(x))) * self.scaling
        return base_out + lora_out


def get_parent_module(model, module_name):
    parts = module_name.split('.')
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]


def apply_lora_to_videomae(model, r=8, alpha=16, dropout=0.1, target_keywords=('query', 'value')):
    replaced = []
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and any(k in name for k in target_keywords):
            parent, child_name = get_parent_module(model, name)
            setattr(parent, child_name, LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
            replaced.append(name)
    print(f'LoRA applied to {len(replaced)} Linear layers')
    print(replaced[:10], '...' if len(replaced) > 10 else '')
    return model


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

## Cell 11 — Model loader: FullFT or LoRA

In [ ]:
def load_videomae_model(num_classes, label2id, id2label, ft_type):
    model = VideoMAEForVideoClassification.from_pretrained(
        MODEL_CKPT,
        num_labels=num_classes,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,
    )

    if ft_type == 'FullFT':
        for p in model.parameters():
            p.requires_grad = True

    elif ft_type == 'LoRA':
        for p in model.parameters():
            p.requires_grad = False
        model = apply_lora_to_videomae(
            model,
            r=CONFIG['lora_r'],
            alpha=CONFIG['lora_alpha'],
            dropout=CONFIG['lora_dropout'],
        )
        # Train the new classifier head too.
        for p in model.classifier.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f'Unknown ft_type: {ft_type}')

    total, trainable = count_params(model)
    print(f'Model loaded: VideoMAE | {ft_type} | classes={num_classes}')
    print(f'Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({100*trainable/total:.2f}%)')
    return model

## Cell 12 — Trainer helper and one-experiment runner

In [ ]:
def make_training_args(output_dir, lr):
    args_kwargs = dict(
        output_dir=output_dir,
        remove_unused_columns=False,
        logging_strategy='steps',
        logging_steps=20,
        learning_rate=lr,
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['grad_accum'],
        num_train_epochs=CONFIG['epochs'],
        warmup_ratio=0.1,
        weight_decay=CONFIG['weight_decay'],
        fp16=torch.cuda.is_available(),
        report_to='none',
        seed=SEED,
        dataloader_num_workers=2,
        save_strategy='no',
    )
    # transformers compatibility
    sig = inspect.signature(TrainingArguments.__init__)
    if 'eval_strategy' in sig.parameters:
        args_kwargs['eval_strategy'] = 'epoch'
    else:
        args_kwargs['evaluation_strategy'] = 'epoch'
    return TrainingArguments(**args_kwargs)


def run_one_experiment(split_n, ft_type):
    print("\n" + "="*80)
    print(f"VideoMAE | {ft_type} | split {split_n}")

    classes, label2id, id2label, train_samples, val_samples = load_swin_split_for_videomae(split_n)
    train_dataset = VideoClassificationDataset(train_samples, image_processor, CONFIG['num_frames'])
    val_dataset   = VideoClassificationDataset(val_samples, image_processor, CONFIG['num_frames'])

    model = load_videomae_model(len(classes), label2id, id2label, ft_type)
    lr = CONFIG['lr_full'] if ft_type == 'FullFT' else CONFIG['lr_lora']
    output_dir = f'{LOG_DIR}/videomae_split{split_n}_{ft_type.lower()}'
    training_args = make_training_args(output_dir, lr)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    # Fine-tuning metrics
    train_monitor = PowerMonitor(interval=0.5)
    train_monitor.start()
    train_result = trainer.train()
    ft_metrics = train_monitor.stop()
    print('Train result:', train_result)
    print('FT metrics:', ft_metrics)

    # Inference/evaluation metrics
    pred_monitor = PowerMonitor(interval=0.5)
    pred_monitor.start()
    pred = trainer.predict(val_dataset)
    inf_metrics = pred_monitor.stop()
    print('Prediction metrics:', pred.metrics)
    print('Inference metrics:', inf_metrics)

    # Metrics from Trainer.predict are prefixed with test_ by default.
    acc = pred.metrics.get('test_accuracy', pred.metrics.get('eval_accuracy', None))
    top5 = pred.metrics.get('test_top5_accuracy', pred.metrics.get('eval_top5_accuracy', None))

    result = {
        'model': 'VideoMAE',
        'ft_type': ft_type,
        'split': split_n,
        'accuracy': round(float(acc) * 100, 2) if acc is not None else None,
        'top5_accuracy': round(float(top5) * 100, 2) if top5 is not None else None,
        'ft_latency_s': ft_metrics['latency_s'],
        'ft_avg_power_w': ft_metrics['avg_power_w'],
        'ft_peak_memory_mb': ft_metrics['peak_memory_mb'],
        'ft_energy_j': ft_metrics['energy_j'],
        'inf_latency_s': inf_metrics['latency_s'],
        'inf_avg_power_w': inf_metrics['avg_power_w'],
        'inf_peak_memory_mb': inf_metrics['peak_memory_mb'],
        'inf_energy_j': inf_metrics['energy_j'],
    }

    print('RESULT:', result)

    # Cleanup
    del trainer, model, train_dataset, val_dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Cell 13 — Run experiments

This saves after every single run, so if Colab stops, completed runs stay in JSON. It also skips completed runs automatically.

In [ ]:
# Load existing results if any
if os.path.exists(OUT_PATH):
    with open(OUT_PATH, 'r') as f:
        results = json.load(f)
    print(f'Loaded existing results: {len(results)}')
else:
    results = []
    print('Starting new results list')

completed = {(r['model'], r['ft_type'], r['split']) for r in results}
print('Completed:', completed)

# Run both FullFT and LoRA for each split, aligned with VideoSwin split files.
for split_n in CONFIG['splits']:
    for ft_type in ['FullFT', 'LoRA']:
        key = ('VideoMAE', ft_type, split_n)
        if key in completed:
            print(f'Skipping already completed: {key}')
            continue

        result = run_one_experiment(split_n, ft_type)
        results.append(result)
        completed.add(key)

        with open(OUT_PATH, 'w') as f:
            json.dump(results, f, indent=2)
        print(f'Saved {len(results)} results -> {OUT_PATH}')

print('All requested VideoMAE experiments complete or skipped.')

Loaded existing results: 1
Completed: {('VideoMAE', 'FullFT', 10)}
Skipping already completed: ('VideoMAE', 'FullFT', 10)

VideoMAE | LoRA | split 10
Split: 10 classes
Classes: 10
Train samples: 1148
Val samples: 287
Train distribution: Counter({2: 310, 8: 165, 9: 160, 1: 122, 4: 99, 5: 89, 6: 74, 3: 62, 7: 42, 0: 25})
Val distribution: Counter({2: 76, 9: 48, 8: 47, 1: 26, 4: 22, 5: 18, 0: 17, 6: 14, 3: 13, 7: 6})

First train sample:
/content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/data/videos/Spinning something so it continues spinning/40440.webm
Exists: True
Missing among first 50 train samples: 0


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([10, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


LoRA applied to 24 Linear layers
['videomae.encoder.layer.0.attention.attention.query', 'videomae.encoder.layer.0.attention.attention.value', 'videomae.encoder.layer.1.attention.attention.query', 'videomae.encoder.layer.1.attention.attention.value', 'videomae.encoder.layer.2.attention.attention.query', 'videomae.encoder.layer.2.attention.attention.value', 'videomae.encoder.layer.3.attention.attention.query', 'videomae.encoder.layer.3.attention.attention.value', 'videomae.encoder.layer.4.attention.attention.query', 'videomae.encoder.layer.4.attention.attention.value'] ...
Model loaded: VideoMAE | LoRA | classes=10
Trainable: 0.30M / 86.53M (0.35%)


Epoch,Training Loss,Validation Loss,Accuracy,Top5 Accuracy
1,2.078571,2.037644,0.397213,0.829268
2,1.831534,1.811367,0.529617,0.898955
3,1.743094,1.687970,0.554007,0.909408
4,1.685324,1.624850,0.574913,0.919861
5,1.593488,1.604161,0.581882,0.919861


...

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([50])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([50, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


LoRA applied to 24 Linear layers
['videomae.encoder.layer.0.attention.attention.query', 'videomae.encoder.layer.0.attention.attention.value', 'videomae.encoder.layer.1.attention.attention.query', 'videomae.encoder.layer.1.attention.attention.value', 'videomae.encoder.layer.2.attention.attention.query', 'videomae.encoder.layer.2.attention.attention.value', 'videomae.encoder.layer.3.attention.attention.query', 'videomae.encoder.layer.3.attention.attention.value', 'videomae.encoder.layer.4.attention.attention.query', 'videomae.encoder.layer.4.attention.attention.value'] ...
Model loaded: VideoMAE | LoRA | classes=50
Trainable: 0.33M / 86.56M (0.39%)


Epoch,Training Loss,Validation Loss,Accuracy,Top5 Accuracy
1,3.263116,3.315073,0.259952,0.562726


KeyboardInterrupt: 

## Cell 14 — View saved VideoMAE results

In [ ]:
import pandas as pd

with open(OUT_PATH, 'r') as f:
    results = json.load(f)

df = pd.DataFrame(results)
df

,model,ft_type,split,accuracy,top5_accuracy,ft_latency_s,ft_avg_power_w,ft_peak_memory_mb,ft_energy_j,inf_latency_s,inf_avg_power_w,inf_peak_memory_mb,inf_energy_j
0,VideoMAE,FullFT,10,86.41,99.30,3045.45,39.01,2761.81,118814.62,104.07,31.33,1298.23,3260.41
1,VideoMAE,LoRA,10,58.19,91.99,2671.19,37.28,917.12,99592.76,97.34,37.57,861.97,3657.09
2,VideoMAE,FullFT,20,82.89,97.49,6139.39,44.20,3184.36,271374.36,230.92,35.79,1719.38,8265.60
3,VideoMAE,LoRA,20,40.50,80.06,5740.39,33.73,1759.61,193627.65,232.27,31.95,862.42,7419.95
4,VideoMAE,FullFT,50,76.18,94.09,16007.97,42.20,3186.28,675583.22,607.37,35.08,1720.77,21304.52


## Cell 15 — Сombine with existing VideoSwin results



In [ ]:
import pandas as pd, json, os

videoswin_path = f'{RESULTS_DIR}/results_videoswin.json'
combined_path = f'{RESULTS_DIR}/combined_videomae_videoswin_aligned.csv'

all_results = []
if os.path.exists(videoswin_path):
    with open(videoswin_path, 'r') as f:
        all_results.extend(json.load(f))
else:
    print('VideoSwin result file not found:', videoswin_path)

with open(OUT_PATH, 'r') as f:
    all_results.extend(json.load(f))

combined_df = pd.DataFrame(all_results)
combined_df.to_csv(combined_path, index=False)
print('Saved:', combined_path)
combined_df

Saved: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/results/combined_videomae_videoswin_aligned.csv


,model,ft_type,split,accuracy,ft_latency_s,ft_avg_power_w,ft_peak_memory_mb,ft_energy_j,inf_latency_s,inf_avg_power_w,inf_peak_memory_mb,inf_energy_j,top5_accuracy
0,VideoSwin,FullFT,10,68.29,1100.66,66.65,10390.26,73356.89,45.96,66.64,2571.19,3063.07,NaN
1,VideoSwin,LoRA,10,71.43,1054.15,66.20,8783.87,69790.11,46.95,66.89,1522.88,3140.82,NaN
2,VideoSwin,FullFT,20,50.08,2415.06,66.60,10389.75,160838.37,109.65,66.67,2568.62,7310.32,NaN
3,VideoSwin,LoRA,20,58.24,2332.66,66.20,8784.00,154432.35,111.11,67.10,1523.05,7455.42,NaN
4,VideoMAE,FullFT,10,86.41,3045.45,39.01,2761.81,118814.62,104.07,31.33,1298.23,3260.41,99.30
5,VideoMAE,LoRA,10,58.19,2671.19,37.28,917.12,99592.76,97.34,37.57,861.97,3657.09,91.99
6,VideoMAE,FullFT,20,82.89,6139.39,44.20,3184.36,271374.36,230.92,35.79,1719.38,8265.60,97.49
7,VideoMAE,LoRA,20,40.50,5740.39,33.73,1759.61,193627.65,232.27,31.95,862.42,7419.95,80.06
8,VideoMAE,FullFT,50,76.18,16007.97,42.20,3186.28,675583.22,607.37,35.08,1720.77,21304.52,94.09


In [ ]:
import json, os, pandas as pd

RESULTS_PATH = f"{RESULTS_DIR}/results_videomae_aligned.json"

print("Exists:", os.path.exists(RESULTS_PATH))
print("Path:", RESULTS_PATH)

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)

    df = pd.DataFrame(results)
    display(df)
else:
    print("No VideoMAE results file found.")

Exists: True
Path: /content/drive/.shortcut-targets-by-id/1uCTYv0jat3yiqVdhtzszwNnOYWrO9MU2/cs_lab/results/results_videomae_aligned.json


,model,ft_type,split,accuracy,top5_accuracy,ft_latency_s,ft_avg_power_w,ft_peak_memory_mb,ft_energy_j,inf_latency_s,inf_avg_power_w,inf_peak_memory_mb,inf_energy_j
0,VideoMAE,FullFT,10,86.41,99.30,3045.45,39.01,2761.81,118814.62,104.07,31.33,1298.23,3260.41
1,VideoMAE,LoRA,10,58.19,91.99,2671.19,37.28,917.12,99592.76,97.34,37.57,861.97,3657.09
2,VideoMAE,FullFT,20,82.89,97.49,6139.39,44.20,3184.36,271374.36,230.92,35.79,1719.38,8265.60
3,VideoMAE,LoRA,20,40.50,80.06,5740.39,33.73,1759.61,193627.65,232.27,31.95,862.42,7419.95
4,VideoMAE,FullFT,50,76.18,94.09,16007.97,42.20,3186.28,675583.22,607.37,35.08,1720.77,21304.52


In [ ]:
import json

input_path = "/content/drive/MyDrive/VideoMAE_aligned_with_VideoSwin_full_notebook.ipynb"
output_path = "/content/drive/MyDrive/VideoMAE_aligned_clean.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# remove broken widgets metadata
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Saved clean notebook:", output_path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/VideoMAE_aligned_with_VideoSwin_full_notebook.ipynb'

In [ ]:
!find /content -name "*.ipynb" 2>/dev/null